1️⃣ Architecture Overview
User
  ↓
OpenAI Model (Agent)
  ↓
MCP Client (Python)
  ↓
MCP Server (Tool Provider)
  ↓
MySQL Database

CREATE DATABASE company;

USE company;

CREATE TABLE employees (
    id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100),
    department VARCHAR(100),
    salary INT
);

INSERT INTO employees (name, department, salary) VALUES
('Amit Sharma', 'Engineering', 120000),
('Priya Verma', 'Engineering', 95000),
('Rahul Gupta', 'Marketing', 80000),
('Sneha Patil', 'HR', 70000),
('Vikram Singh', 'Engineering', 150000),
('Neha Kulkarni', 'Finance', 110000),
('Arjun Mehta', 'Sales', 90000),
('Pooja Nair', 'Marketing', 85000),
('Rohan Deshmukh', 'Engineering', 105000),
('Kavita Joshi', 'HR', 72000);

Python MCP Tool for MySQL

In [1]:
pip install openai mysql-connector-python fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os

print("Python version:", sys.version)
print("Python executable:", sys.executable)

try:
    from mcp.server.fastmcp import FastMCP
    print("✅ FastMCP is now available!")
except ImportError as e:
    print("❌ FastMCP module not found.")
    print("Error:", e)
    print("\nTry these steps:")
    print("1. Install MCP: pip install mcp")
    print("2. Restart the kernel or Python environment")
    print("3. Verify installation using: pip show mcp")

Python version: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
Python executable: D:\python\python.exe
✅ FastMCP is now available!


import mysql.connector
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("mysql-server")

def run_query(query: str):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="company"
    )

    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)

    result = cursor.fetchall()

    cursor.close()
    conn.close()

    return result


@mcp.tool()
def query_mysql(query: str):
    """Execute a SQL query on MySQL database"""
    return run_query(query)


if __name__ == "__main__":
    mcp.run()

Python Client Using OpenAI + MCP

In [3]:
from openai import OpenAI
from mcp import ClientSession, StdioServerParameters

client = OpenAI()

server = StdioServerParameters(
    command="python",
    args=["employeeserver.py"]
)

async with ClientSession(server) as session:

    tools = await session.list_tools()

    response = client.responses.create(
        model="gpt-4.1",
        input="Show all employees in engineering department",
        tools=tools
    )

    print(response.output_text)

TypeError: ClientSession.__init__() missing 1 required positional argument: 'write_stream'